# Riddick - train on a free Colab GPU

Runs the same `scripts/train.py` / `scripts/train_unsupervised.py` as a local
machine, just on a GPU. CUDA is detected automatically, so no code changes
are needed.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`.

Checkpoints are saved to your Google Drive, because Colab's own disk is wiped
when the session ends.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_ROOT = '/content/drive/MyDrive/Riddick_checkpoints'
os.makedirs(CHECKPOINT_ROOT, exist_ok=True)

## 2. Clone the repo and install

In [ ]:
%cd /content
!rm -rf Riddick
!git clone --depth 1 https://github.com/yanivbahalul/Riddick.git
%cd Riddick
!pip install -q -r requirements.txt

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none - set the GPU runtime!')

## 3. Get the training data

The LOL training pairs are not in the repo (~900MB, already published elsewhere).
Download the archive once from
[Kaggle](https://www.kaggle.com/datasets/alenken/low-light-image-enhancement-datasets)
and put it in your Drive, then point `ARCHIVE` at it.

In [ ]:
ARCHIVE = '/content/drive/MyDrive/archive.zip'
!bash scripts/get_data.sh "{ARCHIVE}"

## 4. Train

The command below is the current best Phase 2 recipe (zero-reference CurveNet).
For Phase 1 (supervised), use `scripts/train.py` instead - see the flags at the
bottom of that file. Give every experiment its own `RUN_NAME` so earlier
checkpoints are never overwritten.

Rough time on a T4: about an hour for 150 epochs. Smoke-test a new config
first with `--epochs 2 --image-size 64`.

In [ ]:
RUN_NAME = 'models_p2'
CHECKPOINT_DIR = f'{CHECKPOINT_ROOT}/{RUN_NAME}'
EPOCHS = 150

TRAIN_CMD = (
    'python -u scripts/train_unsupervised.py'
    f' --epochs {EPOCHS}'
    ' --spatial-weight 1.0 --color-weight 1.0 --tv-weight 40.0'
    ' --exposure-target 0.45'
    f' --checkpoint-dir "{CHECKPOINT_DIR}"'
)
!{TRAIN_CMD}

## 5. Evaluate

PSNR / SSIM on the 15 `eval15` pairs at native resolution. Drop `--curvenet`
for a Phase 1 checkpoint. The result is also saved next to the checkpoint on
Drive.

In [ ]:
eval_output = !python scripts/evaluate.py --curvenet --checkpoint "{CHECKPOINT_DIR}/model_epoch{EPOCHS}.pth"
print('\n'.join(eval_output[-3:]))

with open(f'{CHECKPOINT_DIR}/eval_result.txt', 'w') as f:
    f.write('\n'.join(eval_output))